# Google Merchandise Store: месяц из жизни магазина

В данном тьюториале мы рассмотрим [Google Merchandise Store](https://shop.googlemerchandisestore.com/) – магазин, который продаёт мерч Google, а его [обезличенный кликстрим](https://developers.google.com/analytics/bigquery/web-ecommerce-demo-dataset) используется Google для демонстрации BigQuery. Датасет содержит данные за один месяц – январь 2021: 93К пользователей, 109К сессий, 493К событий.

На примере этого реального датасета мы покажем, как с помощью библиотеки [retentioneering](https://retentioneering.com/docs) можно проводить анализ пользовательского поведения и находить узкие места в продукте.

Обзор воронки в секции 2 приведёт нас к двум исследовательским вопросам:
- Почему теряются пользователи на этапе выбора товара? До листинга товаров доходит половина всех сессий, до карточки товара – 7.8%, а до добавления товара в корзину только 1.3%. Разбираем в секции 3.
- Почему падает конверсия `basket` → `shipping_details`? На этапе чекаута больше всего сессий теряется на первом же шаге: его проходят только 30%, тогда как остальные шаги – более 70%. Разбираем в секции 4.

**Что мы нашли**
- Хаб раздела одежды плохо работает как точка входа. 15% всех сессий магазина начинается на нём, но 88% из них заканчиваются сразу же (против 56% у сессий, начавшихся с подстраницы того же раздела), а покупают они в 20 раз реже. Мы покажем, что дело не в самой странице как таковой (если с неё не начинается сессия, то она конвертирует не хуже других), и не в источниках трафика (по UTM-меткам он такой же, как у других лэндингов), а в том, что пользователи, приземляясь на неё, ожидают увидеть другой контент.
- Больше половины сессий с корзиной (54%) не начинают чекаут вообще: они не доходят ни до входа в аккаунт, ни до ввода адреса. Это главный ограничитель конверсии, и лежит он за пределами формы оформления заказа.
- Внутри чекаута всё падение создаёт вход в аккаунт. 15% сессий с корзиной доходят до `sign_in` и останавливаются там, не сделав ни одной покупки. При этом тем, кто прошёл вход, он покупать не мешает: конверсия около 60% и у авторизованных, и у вошедших. Также наблюдается пониженная конверсия у только что зарегистрировавшихся пользователей – 34.7%, и это отдельная точка роста.

Мы покажем, как можно прийти к этим выводам с помощью методов retentioneering, симулируя рассуждения и работу аналитика, какими они могли бы быть при решении реальных продуктовых задач.

## 1. Подготовка и обзор данных

- Если вам нужен исходный датасет, скачайте [`gms.csv.gz`](https://drive.google.com/file/d/19tAZNl6IROsqXaZlTLWX3WMSIUuDzh-J/view?usp=drive_link)
  и положите рядом с ноутбуком. Схема данных описана в [README](https://github.com/retentioneering/retentioneering-tools/tree/master/notebooks/GMS/README.md).
- Если вам интересно, как адаптировать данные из BigQuery к retentioneering или вы хотите узнать, как именно мы подготавливали данные для анализа, откройте `gms_data_preparation.ipynb`. Для выгрузки данных из BigQuery потребуется указать свой `PROJECT_ID`.

### 1.1 Загрузка исходных данных

In [1]:
import pandas as pd
import retentioneering as rete

print("retentioneering", rete.__version__)

df = pd.read_csv("gms.csv.gz", compression="gzip", sep=";")
df["page_location"] = "https://shop.googlemerchandisestore.com" + df["page_location"]
df.head()

retentioneering 5.2.2


,user_id,session_id,event,timestamp,session_number,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location
0,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:34.089387,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
1,1.000223e+09,1000223163.8035208_1,main,2021-01-07 18:35:39.094280,1,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
2,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
3,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,1,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/
4,1.000436e+07,10004358.089772267_1,PLP: Bags,2021-01-08 04:10:12.372423,1,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...


### 1.2 Загрузка данных в Eventstream

[Eventstream](https://retentioneering.com/docs/eventstream) — центральный класс библиотеки retentioneering.
По сути он является тонкой обёрткой `pandas.DataFrame`, которая знает, что значит каждая колонка. Объявим это через
[схему](https://retentioneering.com/docs/eventstream#schema):

- **`path_cols`** регулирует что считать одной траекторией. Обычно это колонки вроде `user_id` и `session_id`. В исходных данных есть и то, и другое, поэтому в `path_cols` добавляем обе эти колонки: `user_id` для поведения за всё время и `session_id` для одного визита. Первый становится значением по умолчанию; любой метод принимает `path_col="session_id"`, чтобы переключиться.
- **`segment_cols`** — категориальные метки событий, разбивающие ивентстрим на группы, поведение которых хочется сравнивать в ходе анализа. В исходном датасете это `country`,`utm_source`, `utm_medium`, `utm_campaign`, `product_type`.
- **`custom_cols`** — дополнительные данные, которые потребуются для анализа. В нашем случае это `page_location`, который очень пригодится в кейсе 1 (см. секцию 3.2).

Колонки `event` и `timestamp` уже названы дефолтными именами, так что упоминать в схеме их не нужно.

In [2]:
SCHEMA = {
    "path_cols": ["user_id", "session_id"],
    "segment_cols": [
        "device", "country", "utm_source", "utm_medium", "utm_campaign", "product_type",
    ],
    "custom_cols": ["page_location"],
}

stream0 = rete.Eventstream(df, schema=SCHEMA)
stream0.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,subindex,index
0,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2,1
1,1.000300e+06,1000299.7413851356_1,main,2021-01-20 11:05:05.002377,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,raw,2,2
2,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,1
3,1.000557e+06,1000557.2911835024_1,view_promotion,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2
4,1.000557e+06,1000557.2911835024_1,main,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,3


К исходным данным можно обратиться через аксессор `Eventstream.df`. Как мы видим, к оригинальным данным добавились три технические колонки: `event_type`, `index`, `subindex`.

Первый метод, который стоит вызвать в анализе – [`Eventstream.describe()`](https://retentioneering.com/docs/eventstream#describe). Он покажет базовую информацию о загруженном ивентстриме: размер, диапазон дат, частоты событий, распределение длин траекторий, уровни сегментов.

In [3]:
overview = stream0.describe(top_events=12)

print(overview["shape"])
print(overview["date_range"])
display(overview["path_stats"]["user_id"])
display(overview["event_frequency"])

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 47}
{'min': Timestamp('2021-01-01 00:00:08.136569'), 'max': Timestamp('2021-01-31 23:59:55.412363'), 'span': Timedelta('30 days 23:59:47.275794')}


,length,duration
count,92983.000000,9.298300e+04
mean,5.306518,3.612019e+04
std,10.437298,1.891713e+05
min,1.000000,0.000000e+00
25%,2.000000,4.894917e+00
50%,2.000000,5.045870e+00
75%,5.000000,6.255278e+01
90%,10.000000,3.211639e+03
99%,49.000000,1.129364e+06
max,593.000000,2.614472e+06


,event,count,share
0,main,116795,0.236707
1,PLP: Apparel,65391,0.132527
2,view_promotion,52209,0.105811
3,PDP: Apparel,36969,0.074925
4,PLP: Shop By Brand,25923,0.052538
5,store,23871,0.048379
6,PLP: Lifestyle,21323,0.043215
7,basket,18680,0.037859
8,add_to_cart,15179,0.030763
9,select_item,9949,0.020164


Две вещи, которые стоит заметить перед тем, как двигаться дальше:

- 47 уникальных событий, но большинство из них `PLP: Apparel`, `PDP: Bags`, ... — это два типа
  страниц Product List Page и Product Details Page, размноженные по 14 товарным категориям.
- Большинство траекторий очень короткие: медианная длина траекторий – 2 события.  

## 2. Обзор пользовательского поведения

### 2.1 Наивный первый шаг

Давайте попробуем сразу построить один из наиболее важных виджетов retentioneering – [transition graph](https://retentioneering.com/docs/widgets/transition-graph) – и убедимся, что его прямолинейное применение не даст желаемого результата.

In [4]:
stream0.transition_graph()

В таком графе сложно что-то разобрать. Даже в режиме [Auto edge filter](https://retentioneering.com/docs/widgets/transition-graph#edgefilter), который оставляет по три самых частотных исходящих ребра на вершину, 47 уникальных событий дают запутанный клубок. В [гайде по анализу траекторий](https://retentioneering.com/docs/path-analysis#when-the-picture-is-unreadable)
перечислены типичные приёмы для получения более понятной картины, здесь нам хватит двух.

### 2.2 Схлопываем `PLP` и `PDP`

`PLP: Bags` и `PLP: Drinkware` — это по сути одна и та же страница листинга товарной категории, но для разных типов товаров. Названия этих типов лежат в колонке `product_type`, так что, склеив названия таких событий в `PLP`, мы ничего не теряем. Аналогично для `PDP` – страницы товарных карточек. Переименовывать события будем с помощью дата процессора [rename_events](https://retentioneering.com/docs/data-processors/rename-events). Итого, 14 событий листинга категорий и 11 событий карточки товара превращаются в два, а всего уникальных событий становится не 47, а 24.

In [5]:
plp_pdp = {e: e.split(":")[0] for e in stream0.df["event"].unique() if e.startswith(("PLP:", "PDP:"))}

list(plp_pdp.items())[:4]

[('PLP: Clearance', 'PLP'),
 ('PLP: Lifestyle', 'PLP'),
 ('PLP: Apparel', 'PLP'),
 ('PDP: Apparel', 'PDP')]

In [6]:
stream = stream0.rename_events(plp_pdp)
stream.describe()["shape"]

{'n_events': 493416, 'n_paths': 92983, 'n_unique_events': 24}

### 2.3 Убиваем дубли просмотров — `collapse_events`

Повторяющиеся события в траектории мы называем петлями, потому что на графе переходов они отображаются как автопереход из события в себя в виде петли. Иногда повторяющиеся события являются индикатором потенциальной проблемы в CJM, но иногда повторение может быть вполне естественным. Сравните: петля на событии `shipping_details` и петля на `add_to_cart`. В первом случае повторное заполнение адреса в чекауте скорее всего означает какую-то проблему, а во втором случае пользователь действительно может несколько раз подряд добавить товар в корзину.

Схлопывать повторяющиеся события в одно событие или нет – это всегда выбор. В данном случае мы проводим анализ сверху вниз, и пока нас больше интересует общая картина, а не детали, и такое схлопывание нам поможет. Реализуем его с помощью дата процессора [`Eventstream.collapse_events()`](https://retentioneering.com/docs/data-processors/collapse-events) с аргументом `loops=True`.

Вторым аргументом вызова является `path_col="session_id"`. Этот параметр есть во всех методах retentioneering. Он позволяет гибко переключаться между представлениями траекторий на разных уровнях: например, как в нашем случае -- между сессиями и полными траекториями пользователей. Выбор `path_col="session_id"` обоснован тем, что в рамках этого анализа мы скорее заинтересованы в том, как пользователи сходятся к покупке на сессионном уровне: продолжительность всего датасета ограничена одним месяцем, на одного пользователя приходится всего 1.17 сессий, поэтому какие-то эволюционные эффекты в поведении увидеть вряд ли получится. Далее в этом ноутбуке мы везде будем использовать `path_col="session_id"`.

In [7]:
stream = stream.collapse_events(loops=True, path_col="session_id")

print(f"{len(stream0.df):,} событий -> {len(stream.df):,} после схлопывания повторов")

493,416 событий -> 325,345 после схлопывания повторов


Таким образом, мы облегчили наш ивентстрим примерно на треть.

> Фильтр, который мы намеренно *не* применили. Обычно хочется выбросить совсем короткие
> траектории с аргументом "сессии из 1-2 событий это шум, их всё равно не проанализировать". В
> нашем случае это не стоит делать. В кейсе 1 мы покажем, что в коротких сессиях и находится один из инсайтов (см. секцию 3).

### 2.4 Тот же граф, теперь читабельный

Теперь граф переходов выглядит так:

In [8]:
stream.transition_graph(path_col="session_id")

Расположение вершин на графе автоматическое, и обычно оно требует ручной доработки. Чтобы сохранить граф полностью, используйте [`state_file`](https://retentioneering.com/docs/widgets#saving-widget-state). Тогда состояние всех его элементов, в том числе расположение вручную расставленных вершин, фильтров и т.д. переживёт перезапуск ноутбука:

```python
stream.transition_graph(state_file="gms_transition_graph.json")
```

Теперь на графе можно разглядеть некоторую структуру CJM. Но чтобы отобразить её явно, воспользуемся отображением пути на графе c помощью механизма [views](https://retentioneering.com/docs/widgets/transition-graph#views):

In [9]:
happy_path = ["PLP", "PDP", "add_to_cart", "basket", "shipping_details", "payment_details", "purchase"]

stream.transition_graph(
    views=[{"name": "Happy path", "focus": {"type": "path", "nodes": happy_path}}],
    view="Happy path",
    path_col="session_id"
)

Несмотря на то, что этот путь на графе назван `Happy path`, это не значит, что он типичный для пользователя: в правом верхнем углу графа показано, что вероятность пройти этот путь (`P(route)`) меньше 0.01%. Поэтому несмотря на то, что нам как сотрудникам магазина хотелось бы, чтобы пользователи проходили этот путь за одну сессию, в реальности большинство пользователей заходят на сайт без прямого намерения совершить покупку. Тем интереснее будет разобраться, что может быть причиной их ухода.

### 2.5 Исследование happy path

Давайте теперь оценим конверсию между шагами happy path с точки зрения [воронки](https://retentioneering.com/docs/widgets/funnel). Шаги воронки упорядоченные и закрытые: сессия попадает на шаг N, только если прошла все предыдущие шаги в нужном порядке.

Разобьём для удобства этот путь на две части: до начала чекаута и после.

In [10]:
stream.funnel(steps=["PLP", "PDP", "add_to_cart"], path_col="session_id", height=500)

Половина всех сессий (54947) доходит до страницы списка товаров, 7.8% – до страницы карточки товара, и только 1.3% что-то добавляет в корзину. В секции 3 мы отдельно рассмотрим вопрос: почему теряются пользователи на этапе выбора товара?

Теперь вторая часть CJM – чекаут:

In [11]:
stream.funnel(
    steps=["basket", "shipping_details", "payment_details", "purchase"],
    path_col="session_id",
    height=460,
)

Основной отток из чекаута происходит на первом же шаге: конверсия в `basket` → `shipping_details` составляет только 30% и это заметно меньше, чем на остальных шагах. Это будет вторым нашим исследовательским вопросом: почему падает конверсия `basket` → `shipping_details`? Ответ на этот вопрос будет в секции 4.

## 3. Почему теряются пользователи на этапе выбора товара?

### 3.1 Где заканчиваются сессии

Чтобы посмотреть, какие шаги предшествовали `path_end`, удобно воспользоваться [Step matrix](https://retentioneering.com/docs/widgets/step-matrix) или [Step Sankey](https://retentioneering.com/docs/widgets/step-sankey) с аргументом `path_pattern="path_end"`, который выровняет траектории по последнему событию.

In [12]:
stream.step_matrix(path_pattern="path_end", path_col="session_id", height=360)

Отсюда мы немедленно видим, что событием, на котором чаще всего происходил обрыв траектории, является `PLP` (39%), с большим отрывом от которого следует `main` (21%). При этом мы замечаем, что за 2-3 шага до конца доминирует событие `path_start`, что скорее всего означает, что чаще всего в рассмотрение попадают короткие траектории вида `path_start` → `PLP` → `path_end`. Запомним это наблюдение.

Похожую информацию можно было получить с помощью transition graph. Удобнее всего узнать о том, какие события предшествовали `path_end` можно с помощью режима [**Ego view**](https://retentioneering.com/docs/widgets/transition-graph#ego-view) с фокусом на вершину `path_end`. В этом режиме граф разворачивает окрестность одного события в Sankey диаграмму и показывает все входящие и исходящие рёбра с весами `proba_in` для входящих рёбер и `proba_out` для исходящих. В случае фокуса на `path_end` покажутся только входящие рёбра, которые и будут соответствовать событиям, завершающим сессию.

In [13]:
stream.transition_graph(
    path_col="session_id",
    views=[{
        "name": "Where sessions end",
        "egoNode": "path_end",                             # открывает модалку Ego view при применении
        "focus": {"type": "node", "event": "path_end"},    # и оставляет узел сфокусированным под ней
    }],
    view="Where sessions end",
    height=560,
)

Итак, мы видим тот же топ событий, завершающих сессию, который мы видели в step matrix: `PLP` (39%), `main` (21%), `PDP` (16%).

### 3.2 Анализ отдельных страниц

Давайте копнём чуть глубже в сторону `PLP` и попробуем разобраться, почему эта страница часто становится последней в сессии пользователя. Выше мы намеренно схлопывали все страницы товарных категорий в `PLP`, чтобы получить более понятную высокоуровневую картину. Теперь нам нужно наоборот углубиться в конкретные товарные категории, чтобы проверить, нет ли среди них таких страниц, которые влияют на отток больше других. Здесь нам пригодится колонка `page_location`, которую мы обработаем с помощью дата процессора [`urls_to_events`](https://retentioneering.com/docs/data-processors/urls-to-events). Он собирает имена событий из URL, позволяя схлопывать ветки URL-дерева на разных уровнях.

Обратите внимание на порядок: сначала мы разделяем по URL, потом схлопываем повторяющиеся события, как делали это в прошлой секции.

In [14]:
all_pages = (
    stream0
    .urls_to_events("page_location", nodes=[], keep_full_paths=True)
    .collapse_events(loops=True, path_col="session_id")
)

print(f"{all_pages.df['event'].nunique()} различных страниц")

931 различных страниц


In [15]:
all_pages.df.head()

,user_id,session_id,event,timestamp,device,country,utm_source,utm_medium,utm_campaign,product_type,page_location,event_type,index,subindex
0,1.000223e+09,1000223163.8035208_1,main://,2021-01-07 18:35:34.089387,mobile,Australia,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2
1,1.000300e+06,1000299.7413851356_1,main://,2021-01-20 11:04:59.887247,desktop,India,(direct),(none),(direct),NaN,https://shop.googlemerchandisestore.com/,collapsed,1,2
2,1.000436e+07,10004358.089772267_1,PLP: Bags:/google+redesign/bags/backpacks,2021-01-08 04:10:12.372423,mobile,Russia,google,cpc,<Other>,Bags,https://shop.googlemerchandisestore.com/Google...,collapsed,1,2
3,1.000557e+06,1000557.2911835024_1,main://,2021-01-07 12:15:33.038847,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,1,2
4,1.000557e+06,1000557.2911835024_1,view_promotion://,2021-01-07 12:15:38.549142,mobile,India,<Other>,<Other>,<Other>,NaN,https://shop.googlemerchandisestore.com/,raw,2,2


Теперь проанализируем, как каждая страница связана с `path_end` в смысле `proba_in` и `proba_out`, но вместо того, чтобы строить граф, возьмём нужные значения с помощью headless-метода [Eventstream.transition_graph_data()](https://retentioneering.com/docs/widgets/transition-graph#headless-mode). Также добавим общее количество посещений каждой из страниц с помощью метода `Eventstream.get_event_counts()`, чтобы иметь представление, какой трафик проходит через них.

In [16]:
pd.set_option("display.max_colwidth", 70)  # имена событий здесь это URL, дадим им место

# Доли переходов в path_end из каждой страницы
arrivals = all_pages.transition_graph_data(edge_weight="proba_in", path_col="session_id")["path_end"]
# Вероятности перехода из каждой страницы в path_end
exits = all_pages.transition_graph_data(edge_weight="proba_out", path_col="session_id")["path_end"]
# трафик на каждую страницу
visits = pd.Series(all_pages.get_event_counts())

exits = pd.DataFrame({
    "share of all exits": arrivals,
    "visits": visits,
    "exit rate": exits,
}).round(3).sort_values("share of all exits", ascending=False)

display(exits.head(10))
print("Exit rate quantiles:\n", exits["exit rate"].quantile([0.5, 0.75, 0.9]))

,share of all exits,visits,exit rate
main://,0.212,72451.0,0.320
PLP: Apparel:/google+redesign/apparel,0.154,22986.0,0.733
view_promotion://,0.095,44431.0,0.233
PDP: Apparel:/google+redesign/apparel/google+dino+game+tee,0.065,7722.0,0.922
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube,0.055,9458.0,0.630
store:/store.html,0.049,16470.0,0.325
PLP: Apparel:/google+redesign/apparel/mens,0.021,10719.0,0.212
PLP: Lifestyle:/google+redesign/lifestyle/drinkware,0.020,6193.0,0.359
basket:/basket.html,0.019,12328.0,0.171
PLP: Clearance:/google+redesign/clearance,0.018,8897.0,0.222


Exit rate quantiles:
 0.50    0.1300
0.75    0.2860
0.90    0.5362
Name: exit rate, dtype: float64


Таким образом, на одну только страницу `Apparel:/google+redesign/apparel` приходится 15.4% всех завершений сессии, а вероятность завершения сессии, находясь на этой странице, составляет 73.3%. Оба этих высоких показателя заставляют нас думать, что со страницей-хабом одежды `Apparel:/google+redesign/apparel` что-то не так (для сравнения, внутренняя страница `Apparel:/google+redesign/apparel/mens` имеет существенно более низкие показатели: 2.1% и 21.2% соответственно, притом что порядок трафика на неё такой же). Давайте это проверим.

Настолько высокие показатели оттока могут быть связаны с тем, что эта страница является лэндингом для части входящего трафика (напомним, мы не фильтровали короткие траектории), поэтому давайте следующим шагом проверим, сохраняются ли эти показатели настолько же высокими в случаях, когда страница находится не в самом начале траектории, а ниже по течению. Кроме этого, сравним эти показатели с похожими страницами – подстраницами хаба `Apparel`.

### 3.3 Exit rate, разделённый по сценариям

Чтобы сравнить хаб с его же подстраницами, они сначала должны стать двумя разными событиями. `aggregate_children` в `urls_to_events` оставляет URL верхнего уровня `/google+redesign/apparel` отдельным событием и сворачивает все его подстраницы в `sub-page`. Дальше `rename_events` возвращает URL к читаемым именам.

In [17]:
pages = stream0.urls_to_events(
    "page_location",
    nodes=[{"path": "/google+redesign/apparel", "aggregate_children": True, "name": "sub-page"}],
).collapse_events(loops=True, path_col="session_id")

pages.df["event"].value_counts().head(8)

event
main://                                                      72479
view_promotion://                                            44427
PDP: Apparel:/google+redesign/apparel/sub-page               23449
PLP: Apparel:/google+redesign/apparel                        22986
PLP: Apparel:/google+redesign/apparel/sub-page               19336
store:/store.html                                            16475
basket:/basket.html                                          12328
PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube     9458
Name: count, dtype: int64

In [18]:
SPLIT = {
    "PLP: Apparel:/google+redesign/apparel": "Apparel hub",
    "PLP: Apparel:/google+redesign/apparel/sub-page": "Apparel sub-page",
    "PLP: Shop By Brand:/google+redesign/shop+by+brand/youtube": "YouTube hub",
}

# оставляем три разделения выше; у всех остальных отрезаем URL и возвращаем обычное имя
pages = pages.rename_events({e: SPLIT.get(e, e.split(":")[0]) for e in pages.df["event"].unique()})

pages.df["event"].value_counts().head(8)

event
main                72563
PLP                 61900
view_promotion      50623
PDP                 41750
Apparel hub         22986
Apparel sub-page    19336
store               16493
basket              12439
Name: count, dtype: int64

Теперь с помощью [`Eventstream.get_conversion_rate()`](https://retentioneering.com/docs/eventstream#get_conversion_rate) можно посчитать конверсии из некоторого события (якоря) `start_anchor` в наступление `end_anchor="path_end"` на следующем шаге. В качестве `start_anchor` нас будут интересовать следующие [паттерны](https://retentioneering.com/docs/path-patterns):

- `path_start->PAGE` – страница была первым событием сессии;
- `.->PAGE` – перед страницей было некоторое настоящее событие (`.` намеренно не матчит `path_start`).

Аргумент `within=1` потребует, чтобы `end_anchor="path_end"` произошёл ровно на следующем шаге после якоря, задаваемого `start_anchor`. Список из нескольких `start_anchor` посчитает конверсии в `end_anchor` для каждого из них.

Конверсии понимаются именно в смысле траекторий: доля траекторий, в которых встретился `start_anchor`, и потом после которого наступил `end_anchor`.

In [19]:
listing_pages = ["Apparel hub", "Apparel sub-page", "YouTube hub"]
start_events = [{"pattern": f"{lead}->{page}"} for page in listing_pages for lead in ("path_start", ".")]

pages.get_conversion_rate(
    start_anchor=start_events,
    end_anchor="path_end",
    within=1,
    path_col="session_id",
)[["start_anchor", "paths_with_start", "converted", "conversion_rate"]].round(3)

,start_anchor,paths_with_start,converted,conversion_rate
0,path_start->Apparel hub,17392,15292,0.879
1,.->Apparel hub,4340,1351,0.311
2,path_start->Apparel sub-page,2298,1277,0.556
3,.->Apparel sub-page,9835,2331,0.237
4,path_start->YouTube hub,7289,5200,0.713
5,.->YouTube hub,1834,668,0.364


Из этой таблицы следует несколько важных наблюдений:

1. Conversion rate из лэндинговой `Apparel hub` (`path_start->Apparel hub`) в `path_end` (он же bounce rate) ещё выше, чем мы видели ранее: 87.9% вместо 73%, выше, чем у лэндинговых `Apparel sub-page` и `YouTube hub` (55.6% и 71.3%), но главное, что это почти в три раза выше, чем bounce rate для внутреннего `Apparel hub` (`.->Apparel hub`) – 31.1%.
2. У внутренних `Apparel hub` bounce rate примерно такой же, как и у внутренних `Apparel sub-page` и `YouTube hub`.

Получается, что высокий bounce rate объясняется не столько природой самого `Apparel hub`, сколько тем, что на него попадает неудачный трафик. Но давайте продолжим разбираться дальше.

### 3.4 Продвижение по воронке в зависимости от лэндинга

На всякий случай проверим дополнительно, сохраняется ли конверсия из `path_start->Apparel hub` низкой не только с точки зрения быстрого завершения сессии, но и с точки зрения продвижения по воронке. Теперь в качестве `end_anchor` выберем не `path_end`, а поочерёдно `PDP`, `basket`, `purchase`, а также уберём условие `within=1`.

In [20]:
landing_funnel = pages.get_conversion_rate(
    start_anchor=[
        {"pattern": "path_start->Apparel hub"},
        {"pattern": "path_start->Apparel sub-page"},
        {"pattern": ".->Apparel hub"},
        {"pattern": ".->Apparel sub-page"}],
    end_anchor=["PDP", "basket", "purchase"],
    path_col="session_id",
).round(4)

landing_funnel

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,path_start->Apparel hub,PDP,17392,495,0.0285,0.2270,0.1254
1,path_start->Apparel hub,basket,17392,230,0.0132,0.0618,0.2139
2,path_start->Apparel hub,purchase,17392,22,0.0013,0.0100,0.1262
3,path_start->Apparel sub-page,PDP,2298,446,0.1941,0.2270,0.8548
4,path_start->Apparel sub-page,basket,2298,227,0.0988,0.0618,1.5979
5,path_start->Apparel sub-page,purchase,2298,59,0.0257,0.0100,2.5622
6,.->Apparel hub,PDP,4340,1281,0.2952,0.2270,1.3000
7,.->Apparel hub,basket,4340,697,0.1606,0.0618,2.5979
8,.->Apparel hub,purchase,4340,172,0.0396,0.0100,3.9551
9,.->Apparel sub-page,PDP,9835,3743,0.3806,0.2270,1.6763


Видим, что конверсии из `path_start->Apparel hub` в `PDP`, `basket`, `purchase` VS соответствующие конверсии из `path_start->Apparel sub-page` различаются в 7, 7, 20 раз соответственно (0.0285 VS 0.1941, 0.0132 VS 0.0988, 0.0013 VS 0.0257). То есть по мере прохождения по воронке к покупке разница только усугубляется.

При этом показатели lift для `path_start->Apparel hub` меньше 1: то есть старт сессии с `Apparel hub` делает события `PDP`, `basket`, `purchase` в 4-8 раз менее вероятными, в то время как старт с `Apparel sub-page` увеличивает вероятность попадания в `basket` и `purchase` в 1.6 и 2.6 раз соответственно.

Кстати, заметим, что соответствующие отношения lift для `path_start->Apparel hub` VS `path_start->Apparel sub-page` будут давать те же самые значения 7, 7, 20, как и для conversion rate, потому что по определению для любых событий $A_1, A_2, B$ верно:

$$\frac{lift(A_1\rightarrow B)}{lift(A_2\rightarrow B)}=\frac{conversion\_rate(A_1\rightarrow B) / base\_rate}{conversion\_rate(A_2\rightarrow B) / base\_rate}=\frac{conversion\_rate(A_1\rightarrow B)}{conversion\_rate(A_2\rightarrow B)}.$$

Что касается аналогичных сравнений для `.->Apparel hub` VS `.->Apparel sub-page`, различия в conversion rate гораздо менее драматичны, а lift для `.->Apparel hub` везде больше 1.

Из всех этих наблюдений мы выводим гипотезу, что на `Apparel hub` льётся некачественный трафик.

### 3.5 Лэндинг как сегмент

Признак "Началась ли эта сессия на `Apparel hub`" можно сделать не разовым фильтром, а зафиксировать как [сегмент](https://retentioneering.com/docs/segments). Дата процессор [`Eventstream.add_segment()`](https://retentioneering.com/docs/data-processors/add-segment) с аргументом `sql` позволяет гибко назначать уровни сегмента для каждой строки. Мы немного расширим этот признак и зададим сегмент с тремя уровнями в зависимости от первого события в сессии: `Apparel hub`, `Apparel sub-page` или `elsewhere` – они запишутся в колонку `landing`.

In [21]:
pages = pages.add_segment(
    "landing",
    sql="""
        SELECT CASE FIRST_VALUE(event) OVER (PARTITION BY session_id ORDER BY "index")
            WHEN 'Apparel hub' THEN 'Apparel hub'
            WHEN 'Apparel sub-page' THEN 'Apparel sub-page'
            ELSE 'elsewhere'
        END
        FROM eventstream
    """,
    path_col="session_id"
)

Теперь с помощью виджета [Segment overview](https://retentioneering.com/docs/widgets/segment-overview) сравним некоторый набор [метрик](https://retentioneering.com/docs/path-metrics) для уровней этого сегмента. Помимо уже изученных выше конверсий в `PDP`, `basket`, `purchase`, нас будут интересовать значения метрик `length` и `duration`, а также `in_segment_bulk`. Последняя метрика особенно важна, потому что покажет, есть ли различия в источниках трафика между уровнями сегмента. Отдельно заметим, что значения conversion rate, которые мы считали в секции 3.4 можно получить при помощи метрики `has_event`, поскольку предикатные условия `start_event` для `get_conversion_rate` теперь по сути переехали в уровни сегмента `landing`, а значит доля траекторий, содержащих события `PDP`, `basket`, `purchase` и будет численно равна конверсии в эти события.

Обратим также внимание, что поскольку сегмент состоит из трёх уровней, то раскраска строк таблицы в красный и синий не несёт большого смысла: одно из трёх значений всегда будут ярко-красным, и одно всегда ярко-синим.

In [22]:
pages.segment_overview(
    "landing",
    path_col="session_id",
    metrics=[
        {"metric": "length"},
        {"metric": "duration"},
        {"metric": "has_event", "metric_args": {"event": "PDP"}},
        {"metric": "has_event", "metric_args": {"event": "basket"}},
        {"metric": "has_event", "metric_args": {"event": "purchase"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_source"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_medium"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "utm_campaign"}},
        {"metric": "in_segment_bulk", "metric_args": {"segment_name": "device"}},
        {"metric": "in_segment", "metric_args": {"segment_name": "country", "segment_level": ["United States", "India", "Canada", "United Kingdom"]}},
    ],
)

Из распределения метрик `in_segment_bulk_<UTM_TYPE>_any_mean` мы понимаем, что структура трафика внутри уровней сегмента примерно одинакова. Практически по всем  `utm_source`, `utm_medium`, `utm_campaign` их доли сессий внутри уровней сегмента редко различаются более, чем на 3.3 п.п.: максимальная разница между `Apparel hub` и elsewhere по источнику трафика получается для `utm_medium='(data_deleted)'` (`in_segment_bulk_utm_medium_(data deleted)_any_mean` равно 0.012 и 0.045 соответственно).

Это значит, что гипотеза, поставленная в конце 3.4 не подтверждается: трафик, который попадает на `Apparel hub`, по UTM-меткам не отличается. Преобладающим же источником трафика для этого лэндинга является органика, которая потенциально может содержать в себе скрытые паттерны (например, по разным группам поисковых запросов).

In [23]:
hub_sessions = pages.df[pages.df["landing"] == "Apparel hub"].drop_duplicates("session_id")

print("Маркетинговые источники для Apparel hub:")
hub_sessions.groupby(["utm_source", "utm_medium"]).size().sort_values(ascending=False).head(5)

Маркетинговые источники для Apparel hub:


utm_source                       utm_medium
google                           organic       5968
(direct)                         (none)        4127
<Other>                          <Other>       2986
                                 referral      1679
shop.googlemerchandisestore.com  referral       929
dtype: int64

Но при этом есть разница как в конверсиях, так и в длине траекторий (метрики `length_mean`, `duration_mean`). Давайте попробуем копнуть ещё глубже и посмотреть, в чём может быть причина такого различия, если не в источниках трафика.

### 3.6 Различия в первых шагах

Step matrix в [diff-режиме](https://retentioneering.com/docs/widgets#diff-mode) визуализирует пошаговые различия в траекториях двух уровней сегмента. Синие ячейки соответствуют преобладанию данного события на данном шаге для `Apparel hub`, красные – `Apparel sub-page`.

In [24]:
pages.step_matrix(
    diff=("landing", "Apparel hub", "Apparel sub-page"),
    path_col="session_id",
    step_window=5,
    height=380
)

В таком виде Step matrix не даёт никакой новой информации: главное отличие траекторий из `Apparel hub` в том, что они короче (поэтому ячейки с `path_end` подсвечиваются красным). Чтобы увидеть более содержательную картину, мы предварительно обрежем все траектории по второму событию с помощью [Eventstream.truncate_paths()](https://retentioneering.com/docs/data-processors/truncate-paths). Так мы не только удалим короткие траектории, состоящие из одного события, но и уберём различие в первом шаге, которое мы и так знаем: одни идут в `Apparel hub`, а другие – в `Apparel sub-page`.

In [25]:
pages\
    .truncate_paths(start_anchor={"pattern": "path_start->.->.", "at": 2}, end_anchor="path_end", path_col="session_id")\
    .step_matrix(
        diff=("landing", "Apparel hub", "Apparel sub-page"),
        path_col="session_id",
        step_window=5,
        height=380
    )

И здесь появляются более интересные детали. Если траектория прерывается не сразу же, то оказывается, что пользователи с лэндинга `Apparel hub` существенно чаще идут в поиск (события `search` и `view_search_results`), а также в подстраницы `Apparel sub-page`. Это указывает на то, что пользователям что-то не нравится именно в этом лэндинге: они не видят что-то, что они ожидали от этой страницы, идут в поиск, и затем завершают сессию.

К сожалению, сейчас магазин https://shop.googlemerchandisestore.com/ полностью изменил свой дизайн, и сложно сказать, что было не так с `Apparel hub` в январе 2021 года, и чем он принципиально отличался от внутренних страниц `Apparel sub-page`. Можно посмотреть в WayBackMachine ([Google+Redesign/Apparel](https://web.archive.org/web/20210123165920/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel), [Google+Redesign/Apparel/Mens](https://web.archive.org/web/20210304130937/https://shop.googlemerchandisestore.com/Google+Redesign/Apparel/Mens)), но новых гипотез это не даёт.

В реальности, будь мы аналитиками Google, мы могли бы дополнительно посмотреть, по каким запросам приходили пользователи на эти страницы, могли бы посмотреть поисковые запросы внутри магазина, и т.д., но пока всё что мы можем сделать – это констатировать, что пользователи, которые приземляются на `Apparel hub`, ожидают от страницы чего-то другого. Часть из них уходит сразу же, другая часть пытается что-то поискать внутри сайта и отваливается чуть позже. Так что дело здесь не столько в природе самой страницы `Apparel hub`, сколько в том, как она работает как точка входа на сайт.

### 3.7 Итоги кейса 1

Вернёмся к исходному вопросу: почему пользователи теряются на этапе выбора товара?

Мы начали с того, что чаще всего сессия обрывается на листинге товаров: `PLP` – последнее событие в 39% сессий. Углубившись в конкретные страницы, мы нашли, что заметная часть этого оттока приходится на одну страницу – хаб раздела одежды `/google+redesign/apparel`. Дальше мы последовательно проверили три объяснения:

- Страница плохая сама по себе. Не подтвердилось: когда пользователь попадает на хаб изнутри сессии, он ведёт себя примерно как его подстраницы – bounce rate 31% против 24% и 36% у соседей.
- Плохой трафик. Тоже не подтвердилось: по UTM-меткам трафик на хаб не отличается от трафика на другие лэндинги, а преобладает в нём органика.
- Несоответствие ожиданиям. Похоже на правду: те, кто пришёл на хаб извне и не ушёл сразу, заметно чаще идут в поиск и в подстраницы того же раздела – то есть ищут то, чего на странице не нашли.

Итого: проблема не в трафике и не в странице вообще, а в том, как хаб работает именно как точка входа. Пользователь приходит из поиска с конкретным запросом, видит витрину раздела и уходит: 88% таких сессий заканчиваются, не открыв больше ни одной страницы, против 56% у сессий, начавшихся с подстраницы. До покупки они доходят в 20 раз реже.

Что с этим делать продукту: пересмотреть, что видит человек, попавший на хаб извне, и проверить по каким поисковым запросам он туда приходит.

## 4. Потери в конверсии чекаута

Давайте теперь разберёмся, в чём причина падения конверсии в процессе чекаута после шага `basket`, упомянутой в секции 2.5.

### 4.1 Анализ с помощью step matrix + get_conversion_rate

Напомним, что в 6743 сессиях пользователи доходили до корзины, но только в 1092 из них достигалась покупка. Конверсия из `basket` в `shipping_details` составляет 30%, а в покупку -- 16%. Будем называть эти конверсии базовыми и все дальнейшие сравнения проведём относительно них.

In [26]:
stream.get_conversion_rate(start_anchor="basket", end_anchor=["shipping_details", "purchase"], path_col="session_id")

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket,shipping_details,6743,2078,0.308171,0.019096,16.137650
1,basket,purchase,6743,1092,0.161946,0.010020,16.161679


Предварительно заметим, что само по себе это не обязательно сигнализирует о проблеме. Из практики ecom общеизвестно, что среди покупателей распространён паттерн, когда они используют корзину как страницу-закладку с сохранёнными товарами, вовсе не намереваясь совершить покупку (или по крайней мере не в эту сессию).

#### 4.1.1 Что происходит после `basket`

Посмотрим на центрированной step matrix, что происходит после того, как пользователи заходят в корзину.

In [27]:
stream.step_matrix(path_pattern="basket", path_col="session_id")

Оказывается, что самым частотным шагом после `basket` является `sign_in`, составляющий 24%. Это важный переход, потому что в отличие от паттернов, которые мы рассмотрим ниже, этот переход по крайней мере не означает явного отклонения от движения по воронке чекаута на первом же шаге.

Такую же долю – 24% – занимает и переход в `path_end`, но это очевидно траектории, которые и не намеревались завершиться покупкой. 21% уходит в PLP, но и эти траектории также не намеревались сконвертироваться в покупку (или по крайней мере не прямо сейчас: пользователи зашли в корзину и продолжили выбор товара дальше). То же самое касается и перехода в `PDP`, который занимает 10%. А вот переход в `shipping_details`, составляющий 5%, означает две вещи:
- Эти пользователи уже авторизованы,
- Эти пользователи намерены совершить покупку.

Таким образом, нас интересуют два под-паттерна и два связанных с ними вопроса:
- `basket->sign_in`: может ли запрос авторизации снижать конверсию в покупку?
- `basket->shipping_details`: правда ли, что этот переход является индикатором "happy path", то есть ведёт к высокой конверсии в покупку?

Для быстрой проверки по этим вопросам снова воспользуемся `get_conversion_rate`:

In [28]:
stream.get_conversion_rate(
    start_anchor=[
        {"pattern": "basket->sign_in"},
        {"pattern": "basket->shipping_details"}],
    end_anchor="purchase",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in,purchase,2258,681,0.301594,0.01002,30.098176
1,basket->shipping_details,purchase,1589,833,0.524229,0.01002,52.316431


Действительно, `basket->shipping_details` демонстрирует высокую конверсию в 52% -- больше, чем в 3 раза выше базовой, что определённо выражает сильную интенцию к покупке, и с этим случаем всё ясно.

По сравнению с этим паттерном, у `basket->sign_in` конверсия в 30% почти в два раза ниже, однако это всё выше базовой конверсии, почти в два раза. Где же дальше теряются пользователи, последовавшие по этому пути?

#### 4.1.2 Где теряется конверсия после `basket`→`sign_in`



Предыдущее сравнение конверсий из `basket->shipping_details` и `basket->sign_in` в покупку не совсем корректно. Дело в том, что первый паттерн оканчивается `shipping_details`, и это означает, что эти пользователи уже продвинулись глубже по воронке чекаута. Посмотрим, во-первых, сколько сессий с `basket->sign_in` доходит до `shipping_details`, а во-вторых -- сколько из них потом дошло до `purchase`.

In [29]:
stream.get_conversion_rate(
    start_anchor="basket->sign_in",
    end_anchor="shipping_details",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in,shipping_details,2258,1397,0.618689,0.019096,32.398161


In [30]:
stream.get_conversion_rate(
    start_anchor=[
        "basket->sign_in->.*->shipping_details",
        "basket->shipping_details"
    ],
    end_anchor="purchase",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->.*->shipping_details,purchase,1397,681,0.487473,0.01002,48.648305
1,basket->shipping_details,purchase,1589,833,0.524229,0.01002,52.316431


Теперь мы видим, что если пользователи в сессии дошли до `shipping_details` после `basket->sign_in`, то дальше их конверсия в покупку не сильно отличается от тех, кто пошёл в `shipping_details` сразу же после `basket` (48.7% против 52.4% соответственно). А вот конверсия из `basket->sign_in` в `shipping_details` составляет всего 61.8%. Таким образом, исследуемая низкая конверсия $CR(\text{basket->sign\_in}, \text{purchase})$ раскладывается так:

$$
CR(\text{basket->sign\_in}, \text{purchase}) = CR(\text{basket->sign\_in}, \text{shipping\_details}) \cdot CR(\text{basket->sign\_in->.*->shipping\_details}, \text{purchase}) = 0.487473 * 0.618689 = 0.30159
$$

То есть основные потери в этой ветке траекторий происходят из-за того, что пользователи не доходят до `shipping_details`.

Попробуем дальше проследить, почему это происходит. Для этого построим step matrix с паттерном `basket->sign_in->[^shipping_details]*->path_end` -- сессии, в которых был переход `basket->sign_in`, после которого не последовало `shipping_details`.

In [31]:
stream.step_matrix(path_pattern="basket->sign_in->[^shipping_details]*->path_end", path_col="session_id")

Видно, что после `basket->sign_in` сессии заканчивались либо сразу же на `path_end` (31%), либо уходили в `registration` (19%), либо уходили из флоу чекаута совсем: `basket` 18%, `PLP` 9%, `store` 7%, `view_promotion` 8%. Посмотрим, как эти переходы влияли на конверсию в `shipping_details`.

In [32]:
stream.get_conversion_rate(
    start_anchor=[
        "basket->sign_in->registration",
        "basket->sign_in->basket",
        "basket->sign_in->PLP",
    ],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,shipping_details,795,625,0.786164,0.019096,41.168096
1,basket->sign_in->registration,purchase,795,214,0.269182,0.010020,26.863565
2,basket->sign_in->basket,shipping_details,222,59,0.265766,0.019096,13.917042
3,basket->sign_in->basket,purchase,222,27,0.121622,0.010020,12.137460
4,basket->sign_in->PLP,shipping_details,102,14,0.137255,0.019096,7.187465
5,basket->sign_in->PLP,purchase,102,4,0.039216,0.010020,3.913604


Таким образом из всех этих подмножеств траекторий только паттерн `basket->sign_in->registration` указывает на намерение пойти дальше по воронке чекаута: конверсия в `shipping_details` там довольно высокая 78%, а во всех остальных паттернах существенно ниже. При этом значение в 78% довольно далеко от 100%, а конверсия из `basket->sign_in->registration` в покупку в 26.9% всё же говорит о том, что в этой ветке траекторий есть видимые потери.

#### 4.1.3 Почему падает конверсия после `basket->sign_in->registration`?

Копнём дальше и посмотрим, что же происходило после паттерна `basket->sign_in->registration`. Но прежде чем строить step matrix давайте зададим себе вопрос: какая траектория была бы естественной для гладкого CJM? Наверно мы бы хотели, чтобы после регистрации пользователь автоматически вернулся во флоу чекаута и вскоре мы бы ожидали увидеть событие `shipping_details`. Теперь давайте посмотрим, что же происходит на самом деле.

In [33]:
stream.step_matrix(path_pattern="basket->sign_in->registration", path_col="session_id", step_window=7)

Довольно отчётливо видно, что после регистрации пользователи выполняют следующие действия: `store`, `view_promotion`, `basket`, и только после этого переходят в `shipping_details`. Событие `store` является некоторым алиасом главной страницы с последующим (автоматическим?) показом `view_promotion`. То есть получается, что магазин после успешной регистрации редиректит пользователя на главную, и пользователь вынужден возвращаться в корзину, чтобы продолжить оформлять заказ.

И теперь давайте проверим, в какой мере это отклонение от чекаута сбивает с покупки. Ему будет соответствовать паттерн `basket->sign_in->registration->[store|view_promotion]*->basket`.

In [34]:
stream.get_conversion_rate(
    start_anchor=[
        {"pattern": "basket->sign_in->registration"},
        {"pattern": "basket->sign_in->registration->[store|view_promotion]*->basket"}],
    end_anchor=["shipping_details", "purchase"],
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,shipping_details,795,625,0.786164,0.019096,41.168096
1,basket->sign_in->registration,purchase,795,214,0.269182,0.010020,26.863565
2,basket->sign_in->registration->[store|view_promotion]*->basket,shipping_details,549,499,0.908925,0.019096,47.596618
3,basket->sign_in->registration->[store|view_promotion]*->basket,purchase,549,180,0.327869,0.010020,32.720292


Как видно, такое отклонение даёт конверсию в 32.7% в покупку и 90.8% в `shipping_details`. То есть мы не можем утверждать, что оно совсем уж негативно влияет на CJM: из 795 траекторий с `basket->sign_in->registration` 549 повторно заходят в `basket` через редирект на `store` (549 / 795 = 69%), и дальше 499 из них (90.8%) доходит до `shipping_details`. И дальше до покупки доходит уже 32% -- примерно на том же уровне, что и конверсия для предиката `basket->sign_in` (30.1%).

Таким образом, мы не можем утверждать, что такая шероховатость в CJM портит конверсию, хотя всё же было бы лучше редиректить пользователя обратно в корзину -- туда, где мы прервали флоу чекаута предложением авторизоваться и потом зарегистрироваться. Хотя всё же отдельно заметим, что примерно 11% траекторий после регистрации не доходят до корзины: конверсия из `basket->sign_in->registration` в `basket` составляет 89%.

In [35]:
stream.get_conversion_rate(
    start_anchor={"pattern": "basket->sign_in->registration"},
    end_anchor="basket",
    path_col="session_id"
)

,start_anchor,end_anchor,paths_with_start,converted,conversion_rate,base_rate,lift
0,basket->sign_in->registration,basket,795,710,0.893082,0.061818,14.446919


#### 4.1.4 Вклад поведенческих веток в конверсию

Подведём итог предыдущим находкам и сделаем разбиение всех траекторий после `basket` на несколько взаимоисключающих типов и посчитаем их конверсии и объёмы.

1. Ушли, не начав чекаут.
2. Перешли в `shipping_details` без `sign_in` (уже были авторизованы).
3. Перешли в `shipping_details` после `sign_in`, без регистрации (были зарегистрированы, но не были авторизованы).
4. Перешли в `shipping_details` после регистрации.
5. Не дошли до `shipping_details` после регистрации.
6. Потерялись после `sign_in`.

К результатам обычного вызова расчёта конверсии `get_conversion_rate` добавим также следующие колонки:
- share – доля паттерна среди всех траекторий, содержащих `basket`;
- contribution – доля сессий из `converted` среди всех траекторий, содержащих `basket`;
- magnitude – отклонение конверсии паттерна в `purchase` от базовой конверсии (в п.п.), взвешенное по `share`.

И `contribution` и `magnitude` дают два взгляда на разбиение базовой конверсии по паттернам. Принципиальное отличие их в том, что первое не различает между собой паттерны с нулевой конверсией, какими бы крупными по размеру они ни были, а второе более гибко учитывает и величину конверсии, и объём паттерна, поэтому смотреть мы будем именно на `magnitude`.

In [36]:
FIRST = "path_start->[^basket]*->basket"   # привязка к первой корзине сессии
G = "[^sign_in|shipping_details]*"         # по пути не встретились ни вход, ни адрес
H = "[^shipping_details|registration]*"    # после входа не встретились ни регистрация, ни адрес
K = "[^shipping_details]*"                 # после регистрации не встретился адрес

branches = {
    "1 - ушли, не начав чекаут":        f"{FIRST}->{G}->path_end",
    "2 - сразу к shipping_details (была авторизация)": f"{FIRST}->{G}->shipping_details",
    "3 - вход в существующий аккаунт -> shipping_details": f"{FIRST}->{G}->sign_in->{H}->shipping_details",
    "4 - вход -> регистрация -> shipping_details": f"{FIRST}->{G}->sign_in->{H}->registration->{K}->shipping_details",
    "5 - вход -> регистрация -> drop-off":  f"{FIRST}->{G}->sign_in->{H}->registration->{K}->path_end",
    "6 - вход -> drop-off":                 f"{FIRST}->{G}->sign_in->{H}->path_end",
}

res = stream.get_conversion_rate(
    start_anchor=[{"pattern": p} for p in branches.values()],
    end_anchor="purchase",
    path_col="session_id",
)\
.drop(columns=["start_anchor", "end_anchor", "base_rate", "lift"])
res.insert(0, "branch", list(branches))

n, p = res["paths_with_start"].sum(), res["converted"].sum()
cr = p / n
res["share"] = res["paths_with_start"] / n
res["contribution"] = 100 * res["converted"] / n                        # в сумме даёт CR всей выборки
res["magnitude"] = 100 * res["share"] * (res["conversion_rate"] - cr)   # в сумме даёт ноль

total = pd.DataFrame([{
    "branch": "total", "paths_with_start": n, "converted": p, "share": 1.0, "conversion_rate": cr,
    "contribution": 100 * cr, "magnitude": res["magnitude"].sum(),
}])

pd.concat([res.sort_values("magnitude"), total])

,branch,paths_with_start,converted,conversion_rate,share,contribution,magnitude
0,"1 - ушли, не начав чекаут",3642,0,0.000000,0.540116,0.000000,-8.746942
5,6 - вход -> drop-off,807,0,0.000000,0.119680,0.000000,-1.938161
4,5 - вход -> регистрация -> drop-off,216,0,0.000000,0.032033,0.000000,-0.518764
3,4 - вход -> регистрация -> shipping_details,641,223,0.347894,0.095062,3.307133,1.767652
1,2 - сразу к shipping_details (была авторизация),630,377,0.598413,0.093430,5.590983,4.077921
2,3 - вход в существующий аккаунт -> shipping_details,807,492,0.609665,0.119680,7.296456,5.358295
0,total,6743,1092,0.161946,1.000000,16.194572,0.000000


Среди паттернов с нулевой конверсией самым сильным является паттерн `1 - ушли, не начав чекаут` (54% траекторий). Это самая крупная группа, и внутри этих сессий нет сигналов намерения купить: пользователи не доходят ни до `shipping_details`, ни до входа в аккаунт. Изменениями в процессе чекаута здесь вряд ли можно будет добиться улучшения.

Паттерны `6 - вход -> drop-off` (11.9%) и `5 - вход -> регистрация -> drop-off` (3.2%) можно интерпретировать как нежелание пользователей авторизовываться и/или регистрироваться ради покупки. Возможно, для них имеет смысл ввести процесс временной регистрации и спасти эти 15% сессий в сумме, хотя не стоит ожидать высокой конверсии даже в этом случае.
 
Среди положительных паттернов 21% в сумме дают быстрый переход к `shipping_details` либо напрямую после `basket` (паттерн 2, 9.3%), либо с коротким отклонением в `sign_in` (паттерн 3, 11.9%) – оба с высокой конверсией в покупку около 60%; и 9.5% занимает более слабый паттерн `4 - вход -> регистрация -> shipping_details` с конверсией 34.7%.

### 4.2 Поиск причин падения через кластеризацию траекторий

Вы могли заметить, что анализ в предыдущем параграфе получился довольно муторным: нам приходилось несколько углубляться в ветвящиеся траектории и считать конверсию. В качестве альтернативного варианта поиска ответа на вопрос "почему падает конверсия?" можно предложить кластеризовать траектории, которые не дошли до таргетного события. Кластерный анализ даст возможность качественно описать те же траектории по длине и составу событий.

"Не дошли до таргетного события" выражается фильтром паттерна `path_start->[^basket]*->basket->[^shipping_details]*->path_end`, который даёт буквально объединение паттернов 1, 5 и 6 из предыдущей секции 4.1.4. Обратите внимание, что часть `[^basket]*` нужна для того, чтобы не включать траектории, в которых были заходы в корзину с продолжением в `shipping_details` и без -- например `...basket->shipping_details->...->basket->PLP->path_end`.

После этого дополнительно обрежем траектории от `basket` до `path_end`, чтобы снизить шум от событий, происходивших до `basket`. И наконец, применим [cluster analysis](https://retentioneering.com/docs/widgets/cluster-analysis) по метрикам-счётчикам событий (`event_count_bulk`).

In [37]:
(
    stream
    .filter_paths({"metric": "matches_pattern", "metric_args": {"pattern": "path_start->[^basket]*->basket->[^shipping_details]*->path_end"}, "op": "=", "value": True}, path_col="session_id")
    .truncate_paths(start_anchor="basket", end_anchor="path_end", path_col="session_id")
    .cluster_analysis(
        features=[{"metric": "event_count_bulk"}],
        overview_metrics=[
            {"metric": "length"},
            {"metric": "event_count_bulk"},
        ],
        method="kmeans",
        method_args={"n_clusters": "2-8"},
        nmf_components="2-8",
        select={"n_clusters": 3, "nmf_components": 3},
        path_col="session_id"
    )
)

Во вкладке silhouette мы видим, что формально лучшее значение метрики даёт разбиение на два кластера. Однако такое разбиение менее информативно с точки зрения интерпретации: оно лишь отделяет 82 самые длинные сессии от остальных. Поэтому мы выбираем следующее по формальному качеству разбиение на три кластера (с помощью аргумента `select`). Какой же получается интерпретация в этом случае?

Самый большой кластер (91.6%) состоит из коротких траекторий (в среднем 3.244 по метрике `length`), и счётчики многих событий имеют околонулевые значения. За исключением следующих:
- `basket` = 1.218.
- `sign_in` = 0.222
- `store` = 0.186
- `view_promotion` = 0.303
- `main` = 0.17
- `PLP` = 0.501
- `PDP` = 0.267
С `basket` всё понятно: его счётчик больше 1, потому что фильтр по паттерну оставил траектории, где гарантированно есть `basket`. А вот все остальные события соответствуют уходам с траектории чекаута, которые мы уже видели выше в степ-матрице: либо в `PDP`/`PLP`/`main`, либо в `sign_in`/`store`/`view_promotion`.

Во втором по объёму кластере (6.7%) находятся содержательные траектории с большим количеством разнообразных событий. По сути все они соответствуют случаю, когда пользователь проводит сознательную сессию выбора товара в магазине без намерения совершить покупку прямо сейчас.

В третьем кластере (1.7%) находятся сессии со сравнительно большим количеством таких событий как `faq`, `privacy-policy`, `registration`, `return-policy`, `shipping-information`, `terms-of-use`. Таких пользователей можно назвать чувствительными или внимательными, поскольку они изучают много дополнительного материала, связанного с покупкой и доставкой товара.

Таким образом, вместо кропотливого исследования всех ветвящихся траекторий с помощью step matrix, мы получили картину пусть и более грубую количественно, но зато понятную качественно с некоторой оценкой на частоту таких паттернов. Из тех, кто после корзины не дошёл до `shipping_details`:
- 91.6% короткие траектории без признаков намерения совершить покупку, хотя примерно 22% из них сделали хотя бы первый шаг в сторону чекаута – дошли до `sign_in`.
- 6.7% сознательно изучали товар, но не совершили покупку в эту сессию (а значит, вероятно, в следующую сессию можно использовать информацию о том, что это прогретые пользователи).
- 1.7% от продвижения по воронке чекаута, вероятно, удерживают не подходящие им условия покупки и доставки.

### 4.3 Итоги кейса 2

Итак, почему после захода в корзину в сессиях теряется конверсия:

- Больше половины сессий (54%) не проявляют никакого намерения даже начать чекаут: они не доходят ни до входа в аккаунт, ни до заполнения `shipping_details`. Это главный ограничитель базовой конверсии в 16%, и он лежит за пределами самого чекаута.
- Внутри чекаута главная потеря – авторизация/регистрация. Паттерны 5 и 6 дают 15% всех сессий с корзиной и ровно ноль покупок. Это кандидат на A/B-тест гостевого чекаута.
- Пользователи, только что прошедшие регистрацию и вернувшиеся во флоу чекаута – событие `shipping_details` – конвертируются в покупку в 34.7% сессий, и это точка роста. Доля таких сценариев составляет 9.5%, а аналогичная конверсия у уже зарегистрированных пользователей, к которой нужно стремиться, составляет около 60%.

## 5. Заключение

Мы прошли путь типичного продуктового исследования: от общей картины к двум конкретным вопросам и от них – к числам, на которые можно опереться при принятии решения.

Начали мы с прямолинейной попытки использования графа переходов, который оказался нечитаемым, что вынудило нас подготовить данные для анализа: схлопнуть товарные страницы в `PLP` и `PDP`, убрать повторы событий и перейти на сессионный уровень траекторий. Затем воронка happy path показала два узких места, которые и стали исследовательскими вопросами.

- Почему теряются пользователи на этапе выбора товара. Больше трети потерь между листингом и карточкой товара дают сессии, которые начинаются на хабе раздела одежды. Трафик там обычный, и внутри сессии страница работает нормально – она плохо справляется именно с ролью точки входа.

- Почему падает конверсия `basket` → `shipping_details`. Больше половины сессий с корзиной (54%) чекаут даже не начинают, и это главный фактор, снижающий базовую конверсию в 16%. Внутри чекаута всё падение создаёт одна ветка: 15% сессий доходят до входа в аккаунт и останавливаются там, не дав ни одной покупки. А те, кто вход прошёл, покупают одинаково хорошо – около 60%, – кроме только что зарегистрировавшихся пользователей, у которых 34.7%.

Важная оговорка: всё это гипотезы для продукта, а не доказанные причины. Для полноценного доказательства нужно провести A/B-тесты про изменение лэндинга хаба одежды, гостевой чекаут, возврат в корзину после регистрации.

В процессе анализа мы воспользовались многими инструментами retentioneering. С помощью дата процессоров [rename_events](https://retentioneering.com/docs/data-processors/rename-events), [collapse_events](https://retentioneering.com/docs/data-processors/collapse-events), [urls_to_events](https://retentioneering.com/docs/data-processors/urls-to-events), [add_segment](https://retentioneering.com/docs/data-processors/add-segment), [truncate_paths](https://retentioneering.com/docs/data-processors/truncate-paths), [filter_paths](https://retentioneering.com/docs/data-processors/filter-paths) мы приводили данные к нужному уровню детализации и выделяли интересующие траектории или их части. Виджеты [Transition graph](https://retentioneering.com/docs/widgets/transition-graph), [Step matrix](https://retentioneering.com/docs/widgets/step-matrix), [Funnel](https://retentioneering.com/docs/widgets/funnel), [Segment overview](https://retentioneering.com/docs/widgets/segment-overview), [Cluster analysis](https://retentioneering.com/docs/widgets/cluster-analysis) помогали нам увидеть структуру траекторий и сравнить группы пользователей. Особенно пригодился метод [get_conversion_rate](https://retentioneering.com/docs/eventstream#get_conversion_rate), который позволял удобно вычислять конверсию интересующих нас [под-паттернов траекторий](https://retentioneering.com/docs/path-patterns) в покупку и другие события.

Надеемся, что этот тьюториал вдохновил вас на анализ пользовательского поведения, и вы воспользуетесь библиотекой для решения подобных задач в вашей предметной области.